# 🧠 Treinamento Interativo do Modelo NCF e MLflow Tracking

Este notebook demonstra o fluxo de treinamento interativo da rede neural **Neural Collaborative Filtering (NCF)** utilizando o framework PyTorch, com acompanhamento de experimentos via **MLflow**.

--- 
### 1. Importações e Configurações

In [7]:
import json
import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Adicionar a raiz do projeto ao PYTHONPATH para permitir imports do recsys
project_root = (
    Path(__file__).resolve().parents[1]
    if "__file__" in locals()
    else Path(".").resolve().parent
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from recsys.data.dataset import InteractionDataset
from recsys.models.factory import ModelFactory

# Certificar o registro do NCF no Factory
from recsys.models.ncf import NCFRecommender as _  # noqa: F401
from recsys.utils.config import settings

# Fixar seeds para reprodutibilidade
torch.manual_seed(settings.random_seed)
np.random.seed(settings.random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(settings.random_seed)

print("✅ Seeds fixadas em:", settings.random_seed)
print("   Dispositivo ativo:", "CUDA GPU" if torch.cuda.is_available() else "CPU")

✅ Seeds fixadas em: 42
   Dispositivo ativo: CPU


### 2. Carregamento dos Dados
Utilização de dados processados e amostras negativas geradas. Se o diretório `processed` estiver vazio, geram-se dados sintéticos simplificados de simulação no momento da execução para o treinamento.

In [8]:
processed_dir = settings.data_processed_dir
train_path = processed_dir / "train_with_negatives.csv"
val_path = processed_dir / "val.csv"
metadata_path = processed_dir / "metadata.json"

if train_path.exists() and val_path.exists() and metadata_path.exists():
    print("📂 Carregando dados processados do pipeline DVC...")
    with metadata_path.open(encoding="utf-8") as f:
        meta = json.load(f)
    n_users = meta["n_users"]
    n_items = meta["n_items"]
else:
    print("⚠️ Dados processados ausentes. Criando simulação sintética...")
    n_users = 200
    n_items = 100
    n_samples = 5000

    processed_dir.mkdir(parents=True, exist_ok=True)

    # Gerar treino
    train_df = pd.DataFrame(
        {
            "user_id": np.random.randint(0, n_users, size=n_samples),
            "item_id": np.random.randint(0, n_items, size=n_samples),
            "label": np.random.choice(
                [0, 1], size=n_samples, p=[0.75, 0.25]
            ),  # Implícito (com negativos)
        }
    )
    train_df.to_csv(train_path, index=False)

    # Gerar validação
    val_df = pd.DataFrame(
        {
            "user_id": np.random.randint(0, n_users, size=500),
            "item_id": np.random.randint(0, n_items, size=500),
            "label": np.random.choice([0, 1], size=500),
        }
    )
    val_df.to_csv(val_path, index=False)

    meta = {"n_users": n_users, "n_items": n_items, "seed": settings.random_seed}
    with metadata_path.open("w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

print(f"✅ Metadados: n_users={n_users:,} | n_items={n_items:,}")

📂 Carregando dados processados do pipeline DVC...
✅ Metadados: n_users=200 | n_items=100


### 3. PyTorch Dataset e DataLoader

In [9]:
train_ds = InteractionDataset(train_path)
val_ds = InteractionDataset(val_path)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

print(
    f"📦 DataLoaders criados: {len(train_loader)} batches de treino, {len(val_loader)} batches de validação."
)

📦 DataLoaders criados: 20 batches de treino, 2 batches de validação.


### 4. Inicialização do Modelo NCF via ModelFactory

In [10]:
model = ModelFactory.create(
    "ncf",
    n_users=n_users,
    n_items=n_items,
    embedding_dim=16,
    hidden_layers=(32, 16),
    dropout=0.1,
)

print(model)

NCFRecommender(
  (user_embedding): Embedding(200, 16)
  (item_embedding): Embedding(100, 16)
  (mlp): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)


### 5. Loop de Treinamento e Integração com MLflow
Configuração do MLflow para rastreamento dos hiperparâmetros e da perda (BCE Loss) por época.

In [11]:
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCELoss()

epochs = 5
print(
    f"🚀 Iniciando treinamento no MLflow (Experimento: '{settings.mlflow_experiment_name}')"
)

with mlflow.start_run(run_name="ncf-notebook-interactive") as run:
    # Logar hiperparâmetros no MLflow
    mlflow.log_params(model.get_hparams())
    mlflow.log_param("lr", 1e-3)
    mlflow.log_param("epochs", epochs)

    for epoch in range(1, epochs + 1):
        # ── Modo de Treinamento ──
        model.train()
        train_loss = 0.0
        for users, items, labels in train_loader:
            users, items, labels = users.to(device), items.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(users, items)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * users.size(0)
        train_loss /= len(train_loader.dataset)

        # ── Modo de Validação ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for users, items, labels in val_loader:
                users, items, labels = (
                    users.to(device),
                    items.to(device),
                    labels.to(device),
                )
                outputs = model(users, items)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * users.size(0)
        val_loss /= len(val_loader.dataset)

        # Logar métricas de perda no MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)

        print(
            f"   Época {epoch:02d}/{epochs:02d} | Perda Treino: {train_loss:.4f} | Perda Validação: {val_loss:.4f}"
        )

    print(f"✅ Treinamento concluído! Run ID do MLflow: {run.info.run_id}")

🚀 Iniciando treinamento no MLflow (Experimento: 'recsys-movielens')
   Época 01/05 | Perda Treino: 0.6811 | Perda Validação: 0.6935
   Época 02/05 | Perda Treino: 0.6412 | Perda Validação: 0.7052
   Época 03/05 | Perda Treino: 0.5863 | Perda Validação: 0.7607
   Época 04/05 | Perda Treino: 0.5472 | Perda Validação: 0.8561
   Época 05/05 | Perda Treino: 0.5386 | Perda Validação: 0.8640
✅ Treinamento concluído! Run ID do MLflow: cfee6be18a074d3cbc9425719a8708fd


### 6. Geração de Recomendações (Inferência)
Para fins de teste do recomendador, realiza-se o ajuste do histórico do modelo e a geração de recomendações excluindo os itens já visualizados.

In [12]:
# Carregar dados de treino para ajustar o histórico (evitar recomendar itens já vistos)
train_df = pd.read_csv(train_path)
model.fit(user_ids=train_df["user_id"].tolist(), item_ids=train_df["item_id"].tolist())

# Recomendação para o usuário indexado 0
user_test = 0
top_k = model.recommend(user_id=user_test, k=5)
print(f"🎯 Recomendações Top-5 para o Usuário {user_test}: {top_k}")

🎯 Recomendações Top-5 para o Usuário 0: [33, 68, 46, 15, 44]
